# MPS入门: 矩阵乘积态基础

本教程介绍矩阵乘积态(Matrix Product State, MPS)的基本概念和实现。

## 学习目标

1. 理解MPS的张量结构
2. 掌握施密特分解
3. 计算纠缠熵
4. 验证面积律

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../../common')

from utils.tensor_utils import entanglement_entropy, pauli_matrices
from visualization.tn_plots import plot_mps_structure, plot_entanglement_profile

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. MPS表示

一维量子态可以表示为张量乘积:

$$
|\psi\rangle = \sum_{s_1,\ldots,s_L} A^{s_1}[1] A^{s_2}[2] \cdots A^{s_L}[L] |s_1 s_2 \cdots s_L\rangle
$$

其中 $A^{s_i}[i]$ 是 $\chi_{i-1} \times \chi_i$ 矩阵。

In [ ]:
# 可视化MPS结构
L = 6
chi_values = [2, 4, 8, 8, 4, 2]

plot_mps_structure(L=L, chi_values=chi_values)
plt.show()

## 2. 施密特分解

将系统分为左右两部分:

$$
|\psi\rangle = \sum_\alpha \lambda_\alpha |\phi_\alpha\rangle_L \otimes |\chi_\alpha\rangle_R
$$

施密特值 $\lambda_\alpha$ 描述纠缠程度。

In [ ]:
# 示例: 贝尔态的施密特分解
bell_state = np.array([1, 0, 0, 1]) / np.sqrt(2)  # |00⟩ + |11⟩

# Reshape为矩阵
psi_matrix = bell_state.reshape(2, 2)

# SVD
U, S, Vt = np.linalg.svd(psi_matrix)

print("施密特值:", S)
print("\n最大纠缠态: 两个相等的施密特值")

# 计算纠缠熵
S_ent = entanglement_entropy(S)
print(f"\n纠缠熵: {S_ent:.4f}")
print(f"理论值 (最大纠缠): {np.log(2):.4f}")

## 3. 横场伊辛模型

$$
H = -J \sum_i \sigma_i^z \sigma_{i+1}^z - h \sum_i \sigma_i^x
$$

在临界点 $h_c = J$ 发生量子相变。

In [ ]:
# 构建小系统并精确对角化
L_small = 8
J = 1.0
h = 0.5  # 有序相

# 构建哈密顿量 (这里简化实现)
sigma_0, sigma_x, sigma_y, sigma_z = pauli_matrices()

print(f"小系统精确对角化: L={L_small}")
print(f"参数: J={J}, h={h}")
print("\n(完整实现见 src/mps_ising.py)")

## 4. 纠缠熵标度

- **面积律** (有序相): $S \sim$ const
- **对数发散** (临界点): $S \sim \frac{c}{6} \log L$

In [ ]:
# 模拟不同相的纠缠熵
L = 40
positions = np.arange(1, L)

# 有序相: 饱和
S_ordered = 0.5 * np.ones(L - 1) + 0.1 * np.random.randn(L - 1)

# 临界点: 对数
c = 0.5  # 伊辛CFT中心电荷
S_critical = (c / 6) * np.log(positions) + 0.2 * np.random.randn(L - 1)

# 绘图
fig, ax = plt.subplots()
ax.plot(positions, S_ordered, 'o-', label='Ordered (h << J)', markersize=4)
ax.plot(positions, S_critical, 's-', label='Critical (h = J)', markersize=4)

ax.set_xlabel('Position')
ax.set_ylabel('Entanglement Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 练习

1. 修改代码,计算不同h/J比值的纠缠熵
2. 验证临界点的中心电荷 c ≈ 0.5
3. 研究有限尺寸效应

## 下一步

运行 `src/mps_ising.py` 进行完整计算:

```bash
cd ../src
python mps_ising.py --L 50 --scan
```